# Block 07: Rank-k Quality and Noise-Aware Ablation

**Goal**  
Show when rank helps, when isotropic loss fails under colored noise, and how mismatch/jitter change the baseline.

**Checklist alignment**  
Implements E6, E7, and E8.

**Outputs**  
- tables under `results/tables/`
- figures under `results/figures/`
- manifests under `results/manifests/`
- executed notebook copy under `results/notebooks/` when this suite is run with `--execute`

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "experiment_checklist.md").exists() and (candidate / "implementation").exists():
            return candidate
    raise RuntimeError("Could not locate repo root for notebook execution.")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from implementation.notebook_support import *

cfg = CanonicalConfig().validate()
dirs = ensure_results_dirs(cfg)
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

In [ ]:
out = run_block07_ablation_suite(cfg)
display(out["ablation_df"])
display(out["mismatch_df"])
display(out["shift_df"])

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
df = out["ablation_df"]
ax.bar(df["noise_type"], df["relative_improvement"])
ax.set_ylabel("relative improvement")
ax.set_title("Noise-aware vs isotropic loss")
save_figure(fig, dirs["figures"] / "block_07_e6_ablation.png")
plt.close(fig)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
curve = out["mismatch_curve_df"]
ax.plot(curve["cosine_squared"], curve["of_estimate_for_unit_shape"], marker="o")
ax.set_xlabel("cos^2(theta_w)")
ax.set_ylabel("OF estimate for unit-normalized shape")
ax.set_title("Template mismatch bias curve")
save_figure(fig, dirs["figures"] / "block_07_e7_mismatch_curve.png")
plt.close(fig)

In [ ]:
ablation_path = dirs["tables"] / "block_07_e6_ablation.csv"
mismatch_path = dirs["tables"] / "block_07_e7_mismatch.csv"
mismatch_curve_path = dirs["tables"] / "block_07_e7_bias_curve.csv"
shift_path = dirs["tables"] / "block_07_e8_time_shift.csv"
manifest_path = dirs["manifests"] / "block_07_ablation_summary.json"
save_dataframe(out["ablation_df"], ablation_path)
save_dataframe(out["mismatch_df"], mismatch_path)
save_dataframe(out["mismatch_curve_df"], mismatch_curve_path)
save_dataframe(out["shift_df"], shift_path)
save_json({"saved": [str(p.relative_to(REPO_ROOT)) for p in [ablation_path, mismatch_path, mismatch_curve_path, shift_path]]}, manifest_path)
pd.DataFrame([manifest_row("block_07", "theorem-support", str(ablation_path.relative_to(REPO_ROOT)), cfg)])